# 🚆 IRCTC South Central Zone — Menu RAG Assistant (LLM Version)

**Colab-ready notebook for the LLM Structured Chunking RAG pipeline.**

This notebook:
1. Installs all dependencies (BGE-M3, Qdrant, Ollama, Gradio)
2. Starts an Ollama server and pulls `llama3.2:3b` (fast, ~2GB)
3. Builds a Qdrant vector index from the IRCTC menu chunks
4. Builds a deterministic food item index (no regex, no hallucination)
5. Launches a Gradio chatbot UI with a public share link

---
**Architecture:**
```
Query
  ├── Budget question?        → Deterministic math (no LLM)
  ├── Known food item?        → Deterministic index lookup (no LLM)
  ├── Set + meal type?        → BGE-M3 hybrid search → Llama
  ├── Meal type only?         → Scroll ALL meal chunks → Llama
  └── General / cross-meal?  → Scroll ALL 26 chunks → Llama
```

> **Runtime:** Use GPU (`Runtime → Change runtime type → T4 GPU`) for faster embedding.

## Step 1 — Install Dependencies

In [ ]:
%%capture
!pip install -q \
    FlagEmbedding \
    qdrant-client \
    openai \
    gradio \
    python-dotenv \
    rich

print('Dependencies installed.')

## Step 2 — Install & Start Ollama + Pull Model

Ollama runs as a background server inside Colab.
We use `llama3.2:3b` by default (~2GB) — fast on T4.
Change `OLLAMA_MODEL` below if you want a different model.

In [ ]:
OLLAMA_MODEL = "llama3.2:3b"   # Change to "llama3.1:8b" for higher accuracy (needs more RAM)

import subprocess, time, os

# Install Ollama
print('Installing Ollama...')
!curl -fsSL https://ollama.ai/install.sh | sh 2>&1 | tail -5

# Start Ollama server in background
print('Starting Ollama server...')
subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)  # Wait for server to boot

# Pull the model
print(f'Pulling {OLLAMA_MODEL} (this may take a few minutes)...')
!ollama pull {OLLAMA_MODEL}
print('Ollama ready.')

## Step 3 — Upload Menu Data

Upload your `chunks_llm.json` file.
If you have a Google Drive copy, mount it instead (see the commented block).

In [ ]:
import os

# ── Option A: Upload manually ──────────────────────────────────────
from google.colab import files
os.makedirs('/content/data', exist_ok=True)
print('Select chunks_llm.json to upload:')
uploaded = files.upload()
for fname, data in uploaded.items():
    dest = f'/content/data/{fname}'
    with open(dest, 'wb') as f:
        f.write(data)
    print(f'Saved to {dest}')

# ── Option B: Mount Google Drive (comment out Option A first) ──────
# from google.colab import drive
# drive.mount('/content/drive')
# import shutil
# shutil.copy('/content/drive/MyDrive/irctc-sc-rag/data/chunks_llm.json', '/content/data/')
# print('Copied from Drive.')

## Step 4 — Load Models

In [ ]:
import os, sys, re, json, csv, time
from datetime import datetime

os.environ['USE_TF'] = '0'
os.environ['USE_TORCH'] = '1'
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

from FlagEmbedding import BGEM3FlagModel, FlagReranker

print('Loading BGE-M3 embedder...')
embedder = BGEM3FlagModel('BAAI/bge-m3', use_fp16=True, device='cuda' if os.path.exists('/proc/driver/nvidia') else 'cpu')
print('BGE-M3 loaded.')

try:
    print('Loading BGE Reranker...')
    reranker = FlagReranker('BAAI/bge-reranker-base', use_fp16=True)
    print('Reranker loaded.')
except Exception as e:
    reranker = None
    print(f'Reranker skipped: {e}')

## Step 5 — Build Qdrant Vector Index

In [ ]:
import uuid
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance, VectorParams, SparseVectorParams, SparseIndexParams,
    PointStruct, SparseVector,
    NamedVector, NamedSparseVector, Filter, FieldCondition, MatchValue
)

CHUNKS_PATH  = '/content/data/chunks_llm.json'
QDRANT_PATH  = '/content/qdrant_local'
COLLECTION   = 'irctc_sc_menu_llm'
DENSE_DIM    = 1024
BATCH_SIZE   = 8

# ── Load chunks ──
with open(CHUNKS_PATH, encoding='utf-8') as f:
    chunks = json.load(f)
print(f'Loaded {len(chunks)} chunks.')

# ── Create Qdrant collection ──
os.makedirs(QDRANT_PATH, exist_ok=True)
client = QdrantClient(path=QDRANT_PATH)
client.recreate_collection(
    collection_name=COLLECTION,
    vectors_config={'dense': VectorParams(size=DENSE_DIM, distance=Distance.COSINE)},
    sparse_vectors_config={'sparse': SparseVectorParams(index=SparseIndexParams(on_disk=False))},
)
print(f'Collection "{COLLECTION}" created.')

# ── Embed & index ──
texts = [c['chunk_text'] for c in chunks]
points = []

for i in range(0, len(texts), BATCH_SIZE):
    batch_texts  = texts[i:i+BATCH_SIZE]
    batch_chunks = chunks[i:i+BATCH_SIZE]
    output = embedder.encode(batch_texts, batch_size=len(batch_texts), max_length=512,
                             return_dense=True, return_sparse=True, return_colbert_vecs=False)
    for j, (chunk, dense_vec, lex) in enumerate(zip(batch_chunks, output['dense_vecs'], output['lexical_weights'])):
        point_id = str(uuid.uuid5(uuid.NAMESPACE_DNS, chunk['chunk_id']))
        points.append(PointStruct(
            id=point_id,
            vector={
                'dense': dense_vec.tolist(),
                'sparse': SparseVector(indices=[int(k) for k in lex.keys()],
                                       values=[float(v) for v in lex.values()]),
            },
            payload={
                'chunk_id': chunk['chunk_id'],
                'meal_type': chunk['meal_type'],
                'set_number': chunk.get('set_number'),
                'price': chunk.get('price'),
                'chunk_text': chunk['chunk_text'],
            },
        ))
    print(f'  Embedded batch {i//BATCH_SIZE + 1}/{(len(texts)+BATCH_SIZE-1)//BATCH_SIZE}')

client.upsert(collection_name=COLLECTION, points=points, wait=True)
info = client.get_collection(COLLECTION)
print(f'Index built: {info.points_count} points in Qdrant.')

## Step 6 — Core RAG Functions

In [ ]:
# ──────────────────────────────────────────────────────────
# Constants
# ──────────────────────────────────────────────────────────
OLLAMA_BASE_URL = 'http://localhost:11434/v1'
TOP_K_RETRIEVE  = 20
TOP_K_RERANK    = 15
RRF_K           = 60
LOG_FILE_PATH   = '/content/data/query_log.csv'

PRICE_TABLE = {'Morning Tea': 15, 'Evening Snacks': 50, 'Breakfast': 65, 'Lunch & Dinner': 120}

SYSTEM_PROMPT = """You are an assistant for IRCTC Rajdhani/Duronto South Central Zone
Sleeper Class food menu queries.

RULES:
1. Answer ONLY using explicitly stated facts from the context. Do not infer, assume, or generalize.
2. CRITICAL — All sets accuracy rule: When asked whether ALL sets include an item, you MUST
   verify EVERY set in the context before answering. If even one set does NOT list the item,
   the answer is NO.
3. Prices: Morning Tea Rs.15 | Breakfast Rs.65 | Evening Snacks Rs.50 | Lunch & Dinner Rs.120
4. There are 7 rotating sets (Set 1-7) served cyclically across journeys.
5. For specific set questions, give that set's items exactly from context.
6. For general questions (e.g. what's for breakfast), summarise all 7 sets.
7. Mention Jain/Diabetic food only when specifically asked.
8. Mention Masala Tea only when question is about tea or beverages.
9. Be concise. No filler. Lead with the direct answer.
10. If a food item is not found, say it is not on the menu and suggest the closest alternative.
    Only say 'I don't have that information' for completely unrelated questions."""

BOT_INFO_TEXT = """IRCTC South Central Zone Menu Assistant for Rajdhani/Duronto Sleeper Class.
Covers Morning Tea (Rs.15), Breakfast (Rs.65), Evening Snacks (Rs.50), Lunch/Dinner (Rs.120).
7 rotating sets. Jain/diabetic meals available on request."""

MEAL_TYPE_TERMS = {
    'breakfast', 'lunch', 'dinner', 'tea', 'snacks', 'snack',
    'morning tea', 'evening snacks', 'morning', 'evening',
}
STOPWORDS = {
    'and','or','the','with','in','on','at','to','a','an','of','for','is','are',
    'was','be','by','as','from','this','that','items','item','set','sets','menu',
    'meal','meals','serving','served','include','includes','have','has','does','which','what',
}


# ──────────────────────────────────────────────────────────
# Food item index
# ──────────────────────────────────────────────────────────
def _normalize(text):
    n = text.lower().strip()
    for a, b in [('bh','b'),('dh','d'),('gh','g'),('sh','s'),('th','t'),('ph','f')]:
        n = n.replace(a, b)
    n = re.sub(r'ly$','li',n)
    n = re.sub(r'ey$','i',n)
    return n.replace('oo','u').replace('ee','i')

def _add_to_index(index, key, display, meal_type, set_num):
    if key not in index:
        index[key] = {'display': display, 'meal_sets': {}}
    if meal_type not in index[key]['meal_sets']:
        index[key]['meal_sets'][meal_type] = []
    if set_num not in index[key]['meal_sets'][meal_type]:
        index[key]['meal_sets'][meal_type].append(set_num)

def build_food_index(chunks_path):
    """Build lookup: normalized_item -> {meal_type: [set_nums]}"""
    index = {}
    try:
        data = json.load(open(chunks_path, encoding='utf-8'))
    except Exception:
        return index
    for chunk in data:
        set_num   = chunk.get('set_number')
        meal_type = chunk.get('meal_type', '')
        text      = chunk.get('chunk_text', '')
        if not set_num or not meal_type:   # only skip if data is missing
            continue
        for raw in re.split(r'[+,\n•;\-]', text):
            item = re.sub(r'^[^:]+:\s*', '', raw.strip())
            item = re.sub(r'\s+', ' ', item).strip(' .')
            if not item or len(item) < 3:
                continue
            if item.lower() in MEAL_TYPE_TERMS or item.lower() in STOPWORDS:
                continue
            item_norm = _normalize(item)
            if len(item_norm) >= 3:
                _add_to_index(index, item_norm, item, meal_type, set_num)
            for word in item.split():
                wn = _normalize(word)
                if len(wn) >= 4 and wn not in STOPWORDS and wn not in MEAL_TYPE_TERMS:
                    _add_to_index(index, wn, word, meal_type, set_num)
    return index

def lookup_item_in_query(query_text, food_index):
    """Search query for known menu items. Returns answer string or None."""
    ql = query_text.lower()
    if not re.search(r'\b(which|what|where|does|do|is|are|have|has|include|contain|serve|served|available|find)\b', ql):
        return None
    if re.search(r'\brs\.?\s*\d+|\d+\s*rs', ql):
        return None
    q_norm = _normalize(query_text)
    matched = [(k, v) for k, v in food_index.items()
               if re.search(r'\b' + re.escape(k) + r'\b', q_norm)]
    if not matched:
        return None
    matched.sort(key=lambda x: len(x[0]), reverse=True)
    item_norm, data = matched[0]
    item_display = data['display'].title()
    lines = [f'{item_display} is served in:']
    for mt, set_nums in sorted(data['meal_sets'].items()):
        set_nums = sorted(set(set_nums))
        s = 's' if len(set_nums) > 1 else ''
        lines.append(f'  - {mt} (Rs.{PRICE_TABLE.get(mt,"?")}): Set{s} {", ".join(map(str,set_nums))}')
    return '\n'.join(lines)


# ──────────────────────────────────────────────────────────
# Special queries (deterministic — never reaches LLM)
# ──────────────────────────────────────────────────────────
def handle_special_queries(query_text):
    q = query_text.lower()
    # Invalid set numbers
    set_nums = [int(n) for n in re.findall(r'\bset\s*(\d+)\b', query_text, re.IGNORECASE)]
    invalid_sets = [n for n in set_nums if n < 1 or n > 7]
    if invalid_sets:
        return f'There are only 7 rotating sets (Set 1-7). Set {invalid_sets[0]} does not exist.'
    # Jain / diabetic
    if re.search(r'\b(jain|diabetic|special meal|dietary restriction)\b', q):
        return ('Jain and diabetic food can be provided on request. '
                'Please request it from train staff when your meal is served.')
    # Budget detection
    BUDGET_PATTERN = (
        r'rs\.?\s*(\d+)|(\d+)\s*rs\.?'
        r'|(?:with|have|got|for|get|budget|only|just|spend|using'
        r'|under|upto|up to|rupees?|inr|afford|pay|cost|costs)\s+(\d+)'
    )
    m = re.search(BUDGET_PATTERN, q)
    budget = None
    if m:
        budget = int(next(g for g in m.groups() if g is not None))
    else:
        budget_words = ['get','buy','afford','eat','have','pay','cost','order','meal','food']
        if any(w in q for w in budget_words):
            set_strs = {str(n) for n in re.findall(r'\bset\s*(\d+)\b', q, re.IGNORECASE)}
            nums = [int(n) for n in re.findall(r'\b(\d+)\b', q)
                    if n not in set_strs and 1 <= int(n) <= 10000]
            if nums:
                budget = max(nums)
    if budget is not None:
        has_morning      = any(k in q for k in ['morning','breakfast','am']) and not any(k in q for k in ['evening','pm','dinner'])
        has_evening      = any(k in q for k in ['evening','snacks','pm']) and not any(k in q for k in ['morning','breakfast','lunch'])
        has_lunch_dinner = any(k in q for k in ['lunch','dinner','night','noon','midday'])
        cands = list(PRICE_TABLE.items())
        if has_morning:      cands = [(n,p) for n,p in cands if n in ['Morning Tea','Breakfast']]
        elif has_evening:    cands = [(n,p) for n,p in cands if n == 'Evening Snacks']
        elif has_lunch_dinner: cands = [(n,p) for n,p in cands if n == 'Lunch & Dinner']
        affordable = [(n,p) for n,p in sorted(cands, key=lambda x:x[1]) if p <= budget]
        if affordable:
            lines = [f'With Rs.{budget} you can afford:']
            for name, price in affordable:
                lines.append(f'  - {name} (Rs.{price})')
            total = sum(p for _,p in affordable)
            if not has_morning and not has_evening and not has_lunch_dinner:
                if budget < PRICE_TABLE['Evening Snacks']:
                    lines.append('Note: Only enough for Morning Tea.')
                elif budget < PRICE_TABLE['Breakfast']:
                    lines.append('Tip: Morning Tea + Evening Snacks = Rs.65 total.')
                elif budget >= total:
                    lines.append(f'Tip: All of the above costs Rs.{total} combined.')
            return '\n'.join(lines)
        else:
            if has_morning:      return f'With Rs.{budget}, cannot afford morning meals. Tea=Rs.15, Breakfast=Rs.65.'
            elif has_evening:    return f'With Rs.{budget}, cannot afford Evening Snacks (Rs.50).'
            elif has_lunch_dinner: return f'With Rs.{budget}, cannot afford Lunch & Dinner (Rs.120).'
            else:                return f'With Rs.{budget}, cannot afford any meal. Cheapest is Morning Tea at Rs.15.'
    return None


# ──────────────────────────────────────────────────────────
# Embedding & search helpers
# ──────────────────────────────────────────────────────────
def embed_query(model, query_text):
    output = model.encode([query_text], return_dense=True, return_sparse=True,
                          return_colbert_vecs=False, max_length=256)
    dense = output['dense_vecs'][0].tolist()
    d     = output['lexical_weights'][0]
    return {'dense': dense,
            'sparse_indices': [int(k) for k in d.keys()],
            'sparse_values':  [float(v) for v in d.values()]}

def hybrid_search(client, query_emb, query_filter=None, top_k=TOP_K_RETRIEVE):
    dense_hits  = client.search(collection_name=COLLECTION,
                                query_vector=NamedVector(name='dense', vector=query_emb['dense']),
                                query_filter=query_filter, limit=top_k, with_payload=True)
    sparse_hits = client.search(collection_name=COLLECTION,
                                query_vector=NamedSparseVector(
                                    name='sparse',
                                    vector=SparseVector(indices=query_emb['sparse_indices'],
                                                        values=query_emb['sparse_values'])),
                                query_filter=query_filter, limit=top_k, with_payload=True)
    chunk_map = {}
    for rank, hit in enumerate(dense_hits, 1):
        pid = str(hit.id)
        if pid not in chunk_map:
            chunk_map[pid] = {'id': pid, 'text': hit.payload.get('chunk_text',''), 'payload': hit.payload, 'rrf': 0.0}
        chunk_map[pid]['rrf'] += 1 / (RRF_K + rank)
    for rank, hit in enumerate(sparse_hits, 1):
        pid = str(hit.id)
        if pid not in chunk_map:
            chunk_map[pid] = {'id': pid, 'text': hit.payload.get('chunk_text',''), 'payload': hit.payload, 'rrf': 0.0}
        chunk_map[pid]['rrf'] += 1 / (RRF_K + rank)
    result = sorted(chunk_map.values(), key=lambda c: c['rrf'], reverse=True)
    return result[:top_k]

def rerank_chunks(reranker, query_text, chunks, top_n=TOP_K_RERANK):
    if not chunks: return []
    if reranker is None:
        for c in chunks:
            c['score'] = c['rrf']
        return chunks[:top_n]
    scores = reranker.compute_score([[query_text, c['text']] for c in chunks], normalize=True)
    for i, c in enumerate(chunks):
        c['score'] = float(scores[i])
    chunks.sort(key=lambda c: c['score'], reverse=True)
    if chunks:
        top = chunks[0].get('score', 0)
        thresh = top * 0.5 if top >= 0.3 else 0.01
        chunks = [c for c in chunks if c.get('score', 0) >= thresh]
    return chunks[:top_n]

def scroll_all(client, scroll_filter=None):
    results, _ = client.scroll(collection_name=COLLECTION, scroll_filter=scroll_filter,
                               limit=50, with_payload=True, with_vectors=False)
    return [{'id': str(r.id), 'text': r.payload.get('chunk_text',''),
             'payload': r.payload, 'score': 1.0, 'rrf': 1.0} for r in results]

def build_prompt(query_text, chunks):
    blocks = [f"[{i}] {c['payload'].get('meal_type','?')} | Set {c['payload'].get('set_number','Overview')} | "
              f"Rs.{c['payload'].get('price','?')}\n{c['text']}" for i, c in enumerate(chunks, 1)]
    return (SYSTEM_PROMPT + '\n\n--- RETRIEVED CONTEXT ---\n' + '\n\n'.join(blocks) +
            '\n--- END CONTEXT ---\n\nQuestion: ' + query_text + '\n\nAnswer:')

def generate_answer(prompt_text):
    from openai import OpenAI
    llm = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')
    response = llm.chat.completions.create(
        model=OLLAMA_MODEL,
        messages=[{'role': 'user', 'content': prompt_text}],
        temperature=0.1, max_tokens=400,
    )
    return response.choices[0].message.content.strip()


# ──────────────────────────────────────────────────────────
# Main query pipeline
# ──────────────────────────────────────────────────────────
def run_query(query_text, food_index):
    # 1. Deterministic: budget / jain / invalid set
    special = handle_special_queries(query_text)
    if special:
        return special, [], 0

    # 2. Deterministic: food item lookup
    item_answer = lookup_item_in_query(query_text, food_index)
    if item_answer:
        return item_answer, scroll_all(client)[:5], 0

    # 3. Smart retrieval routing
    conditions = []
    set_match  = re.search(r'\bset\s*([1-7])\b', query_text, re.IGNORECASE)
    if set_match:
        conditions.append(FieldCondition(key='set_number', match=MatchValue(value=int(set_match.group(1)))))

    ql = query_text.lower()
    meal_type_filter = None
    if 'morning tea' in ql:                                    meal_type_filter = 'Morning Tea'
    elif 'evening snack' in ql or 'evening' in ql:             meal_type_filter = 'Evening Snacks'
    elif 'breakfast' in ql:                                    meal_type_filter = 'Breakfast'
    elif 'lunch' in ql or 'dinner' in ql:                      meal_type_filter = 'Lunch & Dinner'

    if meal_type_filter:
        conditions.append(FieldCondition(key='meal_type', match=MatchValue(value=meal_type_filter)))

    query_filter = Filter(must=conditions) if conditions else None
    t0 = time.time()

    if set_match and meal_type_filter:
        # Case A: pinpoint (Set N + meal)
        qemb = embed_query(embedder, query_text)
        cands = hybrid_search(client, qemb, query_filter=query_filter)
        retrieved = rerank_chunks(reranker, query_text, cands)
    elif meal_type_filter:
        # Case B: meal type only — scroll ALL chunks for that meal
        retrieved = scroll_all(client, Filter(must=[
            FieldCondition(key='meal_type', match=MatchValue(value=meal_type_filter))
        ]))
    else:
        # Case C: general — scroll all 26 chunks
        retrieved = scroll_all(client)

    ms = (time.time() - t0) * 1000

    if not retrieved:
        retrieved = [{'id': 'bot_info', 'text': BOT_INFO_TEXT,
                      'payload': {'meal_type': 'General', 'set_number': None, 'price': 0},
                      'score': 0.0, 'rrf': 0.0}]

    prompt = build_prompt(query_text, retrieved)
    try:
        answer = generate_answer(prompt)
    except Exception as e:
        answer = f'Error: {e}'

    return answer, retrieved, ms


# ──────────────────────────────────────────────────────────
# Logging
# ──────────────────────────────────────────────────────────
def log_query_to_csv(query_text, answer, chunks):
    file_exists = os.path.exists(LOG_FILE_PATH)
    parts = []
    for i, c in enumerate(chunks, 1):
        if c.get('id') == 'bot_info': continue
        p = c.get('payload', {})
        set_str = f"Set {p.get('set_number')}" if p.get('set_number') else 'Overview'
        parts.append(f"[{i}] {p.get('meal_type','?')} | {set_str} | {c.get('text','')[:60]}")
    os.makedirs(os.path.dirname(LOG_FILE_PATH), exist_ok=True)
    with open(LOG_FILE_PATH, 'a', newline='', encoding='utf-8') as f:
        w = csv.writer(f)
        if not file_exists:
            w.writerow(['Timestamp','Query','Answer','Chunks'])
        w.writerow([datetime.now().strftime('%Y-%m-%d %H:%M:%S'), query_text, answer, '\n'.join(parts)])
    return LOG_FILE_PATH


# Build food index
food_index = build_food_index(CHUNKS_PATH)
print(f'Food item index built: {len(food_index)} entries.')

## Step 7 — Quick Sanity Check (no Gradio)

In [ ]:
test_queries = [
    'I have Rs.50 what can I get?',
    'Which meal includes sambar?',
    'What sets include upma?',
    'Which sets have poha for breakfast?',
    'What is served for breakfast on Set 3?',
]

for q in test_queries:
    answer, chunks, ms = run_query(q, food_index)
    print(f'Q: {q}')
    print(f'A: {answer}')
    print(f'   [{len(chunks)} chunks | {ms:.0f}ms]')
    print()

## Step 8 — Launch Gradio UI

This will print a **public share link** you can open in any browser.

In [ ]:
import gradio as gr

SAMPLE_QUESTIONS = [
    'What is served for breakfast on Set 3?',
    'I have Rs.50 what can I get?',
    'Is there a non-veg option for lunch?',
    'My mother is Jain and diabetic, what options does she have?',
    'What dal is served in Set 5 lunch?',
    'Tell me about Masala Tea',
]

def handle_query(query_text):
    if not query_text or not query_text.strip():
        return 'Please enter a question.', [], None
    answer, chunks, ms = run_query(query_text, food_index)
    rows = [[
        i,
        c['payload'].get('meal_type', '?'),
        str(c['payload'].get('set_number', 'Overview')) if c['payload'].get('set_number') else 'Overview',
        f"Rs.{c['payload'].get('price', '?')}",
        f"{c.get('score', 0):.4f}",
        c['text'][:80] + '...',
    ] for i, c in enumerate(chunks, 1)]
    log_file = log_query_to_csv(query_text, answer, chunks)
    return answer, rows, log_file

with gr.Blocks(title='IRCTC Menu Assistant', theme=gr.themes.Soft()) as demo:
    gr.Markdown('# 🚆 IRCTC South Central Zone — Menu Assistant (LLM Structured Chunks)')
    gr.Markdown('*Rajdhani / Duronto | Sleeper Class | South Central Zone | 7 Rotating Sets*')

    with gr.Row():
        with gr.Column(scale=3):
            query_input = gr.Textbox(label='Your Question', placeholder='Ask about the menu...')
            query_btn   = gr.Button('Ask', variant='primary')
        with gr.Column(scale=1):
            log_file_output = gr.File(label='Download Query Log (CSV)', interactive=False)

    answer_output = gr.Markdown(label='Answer')

    chunks_table = gr.Dataframe(
        headers=['#', 'Meal Type', 'Set', 'Price', 'Score', 'Preview'],
        label='Retrieved Chunks',
        column_widths=['5%', '12%', '8%', '8%', '10%', '57%'],
    )

    with gr.Row():
        for q in SAMPLE_QUESTIONS:
            btn = gr.Button(q, size='sm')
            btn.click(fn=lambda q=q: q, outputs=query_input).then(
                fn=handle_query, inputs=query_input,
                outputs=[answer_output, chunks_table, log_file_output]
            )

    query_btn.click(fn=handle_query, inputs=query_input,
                    outputs=[answer_output, chunks_table, log_file_output])
    query_input.submit(fn=handle_query, inputs=query_input,
                       outputs=[answer_output, chunks_table, log_file_output])

demo.launch(share=True)